In [1]:
#task1

In [2]:
import pandas as pd
import numpy as np
import os
import json
import timeit



In [3]:
dataset_a = [
    {"username": "ali_01", "email": "ali@gmail.com", "age": 25, "signup_date": "2025-01-01"},
    {"username": "leyla_99", "email": "leyla_at_gmail.com", "age": 22, "signup_date": "2025-01-02"}, # Invalid email
    {"username": "murad_x", "email": "murad@company.com", "age": "otuz", "signup_date": "2025-01-03"}, # Age as string
    {"username": "", "email": "anon@secret.com", "age": 19, "signup_date": "2025-01-04"}, # Empty username
    {"username": "vusal_b", "email": "vusal@mail.ru", "age": 28, "signup_date": "2025-01-05"},
    {"username": "gunay_g", "email": "gunay@gmail.com", "age": -5, "signup_date": "2025-01-06"}, # Invalid age
    {"username": "samir_s", "email": "samir@edu.az", "age": 21, "signup_date": "2025-01-07"},
    {"username": "fidan_f", "email": "fidan@code.edu", "age": 24, "signup_date": "2025-01-08"}
]



In [4]:
 # This is data entered directly by the user. The main problems are malformed emails, empty fields, and data of the wrong type . Strict type validation and regex checks are mandatory before the ML pipeline

In [5]:
dataset_b = [
    {"timestamp": "2025-02-01 10:00:01", "service": "auth-api", "level": "INFO", "message": "User logged in"},
    {"timestamp": "2025-02-01 10:05:22", "service": "payment-gw", "level": "ERROR", "message": "Timeout connecting to bank"},
    {"timestamp": "2025-02-01 10:06:00", "service": "auth-api", "level": "INFO", "message": "Token refreshed"},
    {"timestamp": "2025-02-01 10:10:15", "service": "ui-server", "level": "WARNING", "message": "Slow response from backend"},
    {"timestamp": "2025-02-01 10:12:45", "service": "payment-gw", "level": "INFO", "message": "Retry successful"},
    {"timestamp": "2025-02-01 10:15:00", "service": "ui-server", "level": "INFO", "message": "Page loaded"},
    {"timestamp": "2025-02-01 10:18:30", "service": "auth-api", "level": "ERROR", "message": "Invalid credentials"},
    {"timestamp": "2025-02-01 10:20:10", "service": "payment-gw", "level": "WARNING", "message": "High latency detected"},
    {"timestamp": "2025-02-01 10:22:12", "service": "ui-server", "level": "INFO", "message": "User logout"},
    {"timestamp": "2025-02-01 10:25:55", "service": "auth-api", "level": "INFO", "message": "New registration"}
]



In [6]:
 # System generated. The main problem here is verbosity or loss of records when the system crashes. Timestamp alignment and categorical encoding are required before using for ML.

In [7]:
dataset_c = [
    {"sku": "LAP-001", "product_name": "Laptop X", "quantity_in_stock": 15, "warehouse_location": "Baku", "last_updated": "2025-02-28"},
    {"sku": "MOU-002", "product_name": "Wireless Mouse", "quantity_in_stock": 120, "warehouse_location": "Sumqayit", "last_updated": "2025-03-01"},
    {"sku": "MON-003", "product_name": "27-inch Monitor", "quantity_in_stock": 45, "warehouse_location": "Baku", "last_updated": "2025-03-02"},
    {"sku": "KEY-004", "product_name": "Mechanical Keyboard", "quantity_in_stock": 80, "warehouse_location": "Ganja", "last_updated": "2025-03-01"},
    {"sku": "CAB-005", "product_name": "USB-C Cable", "quantity_in_stock": 300, "warehouse_location": "Sumqayit", "last_updated": "2025-03-03"},
    {"sku": "HEA-006", "product_name": "Noise Cancelling Headphones", "quantity_in_stock": 25, "warehouse_location": "Baku", "last_updated": "2025-03-02"}
]

In [8]:
 # Internal inventory database. Usually cleaner, but may have stale data or synchronization issues between different repositories. Consistency check and filtering by last_updated date are required.

In [9]:
def validate_registrations(entries):
    valid_entries = []
    invalid_entries = []
    
    for entry in entries:
        username = entry.get('username', '')
        email = entry.get('email', '')
        age = entry.get('age')
        
        try:
            age_int = int(age)
            is_valid_age = age_int > 0
        except (ValueError, TypeError):
            is_valid_age = False
            
        if username and '@' in email and is_valid_age:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)
            
    return valid_entries, invalid_entries

valid, invalid = validate_registrations(dataset_a)


In [10]:
print(f"Valid: {len(valid)}, Invalid: {len(invalid)}")

Valid: 4, Invalid: 4


In [11]:
#task2

In [12]:
np.random.seed(42)
n_rows = 500
rides = pd.DataFrame({
    'ride_id': range(1, n_rows + 1),
    'driver_id': np.random.randint(1000, 1051, n_rows),
    'distance_km': np.random.uniform(1.0, 30.0, n_rows),
    'fare_usd': np.random.uniform(5.0, 75.0, n_rows),
    'ride_type': np.random.choice(["standard", "premium", "pool"], n_rows),
    'city': np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"], n_rows),
    'timestamp': pd.date_range(start="2025-01-01", periods=n_rows, freq='min')
})

In [13]:
rides.to_csv('rides.csv', index=False)
rides.to_json('rides.json', orient='records', lines=True)
rides.to_parquet('rides.parquet', index=False)

In [14]:
csv_size = os.path.getsize('rides.csv') / 1024
json_size = os.path.getsize('rides.json') / 1024
parquet_size = os.path.getsize('rides.parquet') / 1024

In [15]:
print(f"CSV size: {csv_size:.2f} KB")
print(f"JSON size: {json_size:.2f} KB")
print(f"Parquet size: {parquet_size:.2f} KB")
print(f"Parquet is {csv_size/parquet_size:.1f}x smaller than CSV.")

CSV size: 39.49 KB
JSON size: 72.44 KB
Parquet size: 20.78 KB
Parquet is 1.9x smaller than CSV.


In [16]:
df_csv = pd.read_csv('rides.csv')
df_parquet = pd.read_parquet('rides.parquet')

In [17]:
 # Parquet format has both the smallest size and the most accurate data type preservation. CSV works everywhere but loses types. JSON can have the largest size because of repeating keywords.

In [18]:
#task3

In [19]:
df_large = pd.DataFrame(np.random.randn(200000, 20), columns=[f"feature_{i}" for i in range(20)])

In [20]:
print("Pandas Column Access (A):")
%timeit df_large["feature_0"].mean()

print("\nPandas Row Iteration (B):")
%timeit [df_large.iloc[i]["feature_0"] for i in range(10000)]

Pandas Column Access (A):
1.61 ms ± 136 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Pandas Row Iteration (B):
145 ms ± 28.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
arr = df_large.values
print("\nNumPy Column Access (C):")
%timeit arr[:, 0].mean()


NumPy Column Access (C):
751 μs ± 62.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [22]:
print("\nNumPy Sequential Access (D):")
def sum_rows(data):
    s = 0
    for i in range(10000):
        s += data[i, 0]
    return s
%timeit sum_rows(arr)


NumPy Sequential Access (D):
1.5 ms ± 91.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [23]:
# Pandas has a column-oriented structure, so performing operations on a column (A) is incredibly fast. Traversing rows (B) is very slow due to Python's object creation overhead.
# In NumPy, this difference is reduced because NumPy arrays are stored in memory more compactly and "C-style", but vectorized operations (C) are still always faster than cycles.

In [24]:
#task4

In [25]:
np.random.seed(42)
df_binary = pd.DataFrame(np.random.randn(100000, 5), columns=[f"f{i}" for i in range(5)])
df_binary['category'] = np.random.choice(["alpha", "beta", "gamma"], 100000)

df_binary.to_csv('test.csv', index=False)
df_binary.to_parquet('test.parquet', index=False)

In [26]:
print("Reading CSV:")
%time it_csv = pd.read_csv('test.csv')

print("Reading Parquet:")
%time it_parquet = pd.read_parquet('test.parquet')

Reading CSV:
CPU times: total: 62.5 ms
Wall time: 114 ms
Reading Parquet:
CPU times: total: 15.6 ms
Wall time: 23.1 ms


In [27]:
print("Reading only 'category' from Parquet:")
%time partial_parquet = pd.read_parquet('test.parquet', columns=['category'])

Reading only 'category' from Parquet:
CPU times: total: 15.6 ms
Wall time: 12.4 ms


In [28]:
# Since Parquet is stored columnar, when we want just one column, it reads only that part of the file without touching the rest.
# CSV is line-based text; even if "category" is the last column, it has to read all the preceding commas and numbers to get there.

In [29]:
# task5

In [30]:
flat_df = pd.DataFrame({
    'title': ['Book A', 'Book A', 'Book B', 'Book C', 'Book C', 'Book D', 'Book E', 'Book F', 'Book G', 'Book H', 'Book I', 'Book J'],
    'author': ['Auth 1', 'Auth 1', 'Auth 2', 'Auth 3', 'Auth 3', 'Auth 4', 'Auth 5', 'Auth 1', 'Auth 2', 'Auth 3', 'Auth 6', 'Auth 7'],
    'format': ['PB', 'EB', 'PB', 'PB', 'EB', 'PB', 'PB', 'EB', 'PB', 'PB', 'EB', 'PB'],
    'publisher_name': ['Pub X', 'Pub X', 'Pub Y', 'Pub Z', 'Pub Z', 'Pub X', 'Pub Y', 'Pub X', 'Pub Y', 'Pub Z', 'Pub W', 'Pub W'],
    'publisher_country': ['USA', 'USA', 'UK', 'Germany', 'Germany', 'USA', 'UK', 'USA', 'UK', 'Germany', 'France', 'France'],
    'price': [20, 15, 25, 30, 22, 18, 12, 10, 28, 35, 40, 50]
})

In [31]:
books = flat_df[['title', 'author', 'format', 'price', 'publisher_name']].copy()
publishers = flat_df[['publisher_name', 'publisher_country']].drop_duplicates()


In [32]:
doc_store = []
for title in flat_df['title'].unique():
    subset = flat_df[flat_df['title'] == title]
    doc = {
        "title": title,
        "author": subset['author'].iloc[0],
        "publisher": {"name": subset['publisher_name'].iloc[0], "country": subset['publisher_country'].iloc[0]},
        "editions": subset[['format', 'price']].to_dict('records')
    }
    doc_store.append(doc)

In [33]:
doc_store

[{'title': 'Book A',
  'author': 'Auth 1',
  'publisher': {'name': 'Pub X', 'country': 'USA'},
  'editions': [{'format': 'PB', 'price': 20}, {'format': 'EB', 'price': 15}]},
 {'title': 'Book B',
  'author': 'Auth 2',
  'publisher': {'name': 'Pub Y', 'country': 'UK'},
  'editions': [{'format': 'PB', 'price': 25}]},
 {'title': 'Book C',
  'author': 'Auth 3',
  'publisher': {'name': 'Pub Z', 'country': 'Germany'},
  'editions': [{'format': 'PB', 'price': 30}, {'format': 'EB', 'price': 22}]},
 {'title': 'Book D',
  'author': 'Auth 4',
  'publisher': {'name': 'Pub X', 'country': 'USA'},
  'editions': [{'format': 'PB', 'price': 18}]},
 {'title': 'Book E',
  'author': 'Auth 5',
  'publisher': {'name': 'Pub Y', 'country': 'UK'},
  'editions': [{'format': 'PB', 'price': 12}]},
 {'title': 'Book F',
  'author': 'Auth 1',
  'publisher': {'name': 'Pub X', 'country': 'USA'},
  'editions': [{'format': 'EB', 'price': 10}]},
 {'title': 'Book G',
  'author': 'Auth 2',
  'publisher': {'name': 'Pub Y', 'c

In [34]:
#Flat (Denormalized): Allows for fast analysis because the data is in one place, but repeating the same information can lead to inconsistencies and errors during updates.

#Normalized: Saves memory by eliminating redundant data and allows for large-scale updates to the entire database with a single click.

#Document (JSON): Stores each object as an independent package with its internal details, which allows for fast access to individual records, but complicates general statistical calculations.